# Laboratory 2 — Building a Place Cell from Scratch

A reproducible mathematical experiment: **position → conditional firing rate → spike counts → occupancy-normalized rate map**. Synthetic trajectories are prescribed; this is not a simulation of learned place-cell formation or animal localization.

**Workflow:** choose parameters → define and validate the rate model → simulate **one fixed fine-time motion trajectory** → sample it at the chosen spike-bin duration → compare Poisson counts and rate maps → change one experimental variable at a time.

**Prerequisites:** NumPy and Matplotlib. Use **Run All** in Jupyter or Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ARENA_SIZE = 1.0          # metres, square [0, 1] × [0, 1]
DT = 0.02                 # seconds per spike-count bin; try 0.01
DURATION = 300.0          # seconds; try 60.0
TRAJECTORY_DT = 0.005     # fixed physical motion integration grid (independent of DT)
MOTION_SEED = 2026
SPIKE_SEED = 2027

if DT <= 0 or DURATION <= 0 or TRAJECTORY_DT <= 0:
    raise ValueError("Time steps and duration must be positive.")
n_steps = int(round(DURATION / DT))
if not np.isclose(n_steps*DT, DURATION, atol=1e-9):
    raise ValueError("Choose a DT that divides the duration exactly.")
print(f"{n_steps:,} spike bins of {DT:g} s for {DURATION:g} s.")
print(f"Underlying motion is always simulated at {TRAJECTORY_DT:g} s.")

## 1. Gaussian field: a modeling assumption

For preferred position $\boldsymbol{\mu}$, baseline $r_{\min}$, peak $r_{\max}$, and width $\sigma$ (meters),

$$r(\mathbf{x}) = r_{\min} + (r_{\max}-r_{\min})\exp\bigl[-\|\mathbf{x}-\boldsymbol{\mu}\|^2/(2\sigma^2)\bigr].$$

In [ ]:
center = np.array([0.55, 0.48])  # meters
sigma = 0.14                       # meters
r_max = 25.0                       # Hz
r_min = 0.2                        # Hz

def place_rate(xy, center=center, sigma=sigma, r_max=r_max, r_min=r_min):
    xy = np.asarray(xy, dtype=float)
    dist2 = np.sum((xy - center)**2, axis=-1)
    return r_min + (r_max - r_min) * np.exp(-dist2/(2*sigma**2))

coord = np.linspace(0, ARENA_SIZE, 120)
X, Y = np.meshgrid(coord, coord)
rate_grid = place_rate(np.stack([X,Y], axis=-1))
fig, ax = plt.subplots(figsize=(5,4))
heat = ax.imshow(rate_grid, extent=[0,1,0,1], origin='lower', cmap='inferno')
ax.scatter(*center, marker='+', c='cyan', s=90, label='field center')
ax.set(xlabel='x (m)', ylabel='y (m)', title='Idealized place-cell expected firing rate')
ax.legend(loc='upper right'); fig.colorbar(heat, ax=ax, label='Expected rate (Hz)')
plt.show()

## Reference validation — model properties (parameter-aware)

Check that the Gaussian field reaches its declared peak at the preferred position and decays correctly one spatial standard deviation away. These properties remain true if the student changes the field width or peak rate. These checks are independent of the later choice of time step.

In [ ]:
assert sigma > 0 and 0 <= r_min <= r_max
assert np.isclose(place_rate(center), r_max)
one_sigma = center + np.array([sigma, 0.0])
assert np.isclose(place_rate(one_sigma), r_min + (r_max-r_min)*np.exp(-0.5))
print("Gaussian reference properties verified at current parameters.")

## 2. Generate one fixed underlying exploration trajectory

The underlying walk is integrated on a **fixed 5 ms motion grid**. Its random heading perturbation is specified per **physical time**, using a standard deviation proportional to the square root of the fine integration step. The fine trajectory uses its own reproducible seed and **never depends on the spike-bin time step DT**.

The path is a synthetic correlated random walk with reflected arena boundaries; it is neither measured rodent movement nor a learned robot controller. The displayed positions are interpolated from this single reference path at times 0, DT, 2DT, and so on. Different values of DT therefore sample **the same motion**, rather than generating a new random walk.

In [ ]:
def reference_walk(duration_s, seed, motion_dt=TRAJECTORY_DT, speed=0.16):
    """Generate a fixed fine-grid synthetic trajectory (not coupled to spike DT)."""
    n = int(round(duration_s/motion_dt))
    if not np.isclose(n*motion_dt, duration_s, atol=1e-9):
        raise ValueError("Duration must be divisible by the motion integration step.")
    xy = np.empty((n+1, 2), dtype=float)
    xy[0] = (0.15, 0.15)
    heading = 0.5
    motion_rng = np.random.default_rng(seed)
    # Original 0.18-rad noise per 20 ms, rescaled for a fixed motion interval.
    heading_sd = 0.18*np.sqrt(motion_dt/0.02)
    for j in range(1, n+1):
        heading += motion_rng.normal(0, heading_sd)
        candidate = xy[j-1] + speed*motion_dt*np.array([np.cos(heading), np.sin(heading)])
        for axis in range(2):
            if candidate[axis] < 0:
                candidate[axis] = -candidate[axis]
                heading = np.pi-heading if axis == 0 else -heading
            elif candidate[axis] > ARENA_SIZE:
                candidate[axis] = 2*ARENA_SIZE-candidate[axis]
                heading = np.pi-heading if axis == 0 else -heading
        xy[j] = np.clip(candidate, 0, ARENA_SIZE)
    return np.arange(n+1)*motion_dt, xy

def sample_path(sample_times, reference_times, reference_xy):
    """Interpolate the same underlying motion at arbitrary observation times."""
    return np.column_stack([np.interp(sample_times, reference_times, reference_xy[:,dim])
                            for dim in range(2)])

reference_times, reference_position = reference_walk(DURATION, MOTION_SEED)
sample_times = np.arange(n_steps)*DT
position = sample_path(sample_times, reference_times, reference_position)
assert len(position) == n_steps
assert (position >= 0).all() and (position <= ARENA_SIZE).all()
print(f"Fine-grid motion samples: {len(reference_position):,}; "
      f"sampled spike bins: {len(position):,}")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(reference_position[:,0], reference_position[:,1], lw=0.55,
        color='0.65', label='Fixed underlying trajectory')
ax.scatter(position[::max(1, int(0.4/DT)),0],
           position[::max(1, int(0.4/DT)),1],
           s=7, color='tab:blue', label='Selected sampling times')
ax.set(xlim=(0,1), ylim=(0,1), xlabel='x (m)', ylabel='y (m)',
       title=f'One motion trajectory, observed every {DT:g} s')
ax.set_aspect('equal')
ax.legend(fontsize=8)
plt.show()

## 3. Convert rate to Poisson spike counts

For a bin of duration DT, the teaching approximation is $N[t]\sim\mathrm{Poisson}(r(\mathbf{x}_t)\,\mathrm{DT})$. This uses the rate at the bin start as approximately constant over the bin. For rapidly varying rates or larger bins, the correct Poisson parameter is the **integrated intensity** $\Lambda[t]=\int_t^{t+\mathrm{DT}}r(\mathbf{x}(s))\,ds$.

A **separate spike RNG** prevents drawing different motion trajectories when spike sampling changes. Independently drawn spike trains may still have different totals by chance; compare their expected counts as well as observed counts.

In [ ]:
expected_rate = place_rate(position)
spike_rng = np.random.default_rng(SPIKE_SEED)
spike_counts = spike_rng.poisson(expected_rate * DT)
expected_total = float(np.sum(expected_rate*DT))
print("Total simulated spikes:", int(spike_counts.sum()))
print("Expected total spikes (bin-start approximation): %.2f" % expected_total)
print("Observed mean firing rate: %.2f Hz" % (spike_counts.sum()/DURATION))
print("Expected mean firing rate: %.2f Hz" % (expected_total/DURATION))
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(position[:,0], position[:,1], color='0.75', lw=0.45)
spike_pos = np.repeat(position, spike_counts, axis=0)
ax.scatter(spike_pos[:,0], spike_pos[:,1], c='darkred', s=3,
           alpha=0.45, label='simulated spikes')
ax.set(xlim=(0,1), ylim=(0,1), xlabel='x (m)', ylabel='y (m)',
       title=f'Synthetic Poisson spikes, bin = {DT:g} s')
ax.set_aspect('equal')
ax.legend()
plt.show()

## 4. Occupancy-normalized rate map

A bin visited twice as long generally receives more spikes, even at identical firing rate. Estimate $\hat r_b=K_b/T_b$ with $K_b$ observed spike count and $T_b$ occupancy seconds. Unvisited bins are masked; we also mask bins with less than 0.2 s of occupancy.

In [ ]:
edges = np.linspace(0, ARENA_SIZE, 21)
occupancy_steps, _, _ = np.histogram2d(position[:,0],position[:,1],bins=[edges,edges])
spikes_per_bin, _, _ = np.histogram2d(position[:,0],position[:,1],bins=[edges,edges],weights=spike_counts)
occupancy_seconds = occupancy_steps * DT
min_occupancy = 0.2
rate_estimate = np.divide(spikes_per_bin,occupancy_seconds,
                          out=np.full_like(spikes_per_bin,np.nan),where=occupancy_seconds>=min_occupancy)
valid = np.isfinite(rate_estimate)
print('Coverage of adequately sampled spatial bins: %.1f%%' % (100*valid.mean()))
# Reference map evaluated at bin centers; sample rate-map errors only where the robot visited.
bin_centers = (edges[:-1]+edges[1:])/2
CX,CY = np.meshgrid(bin_centers,bin_centers,indexing='ij')
reference_rate = place_rate(np.stack([CX,CY],axis=-1))
if valid.any():
    mae = np.nanmean(np.abs(rate_estimate-reference_rate))
    print('MAE across adequately sampled bins: %.2f Hz' % mae)
fig,axes = plt.subplots(1,3,figsize=(13,3.8),constrained_layout=True)
for ax, field, title, cmap in zip(axes,
        [occupancy_seconds,reference_rate,rate_estimate],
        ['Occupancy (s)','Ground-truth rate (Hz)','Estimated rate (Hz)'],
        ['viridis','inferno','inferno']):
    im=ax.imshow(field.T,origin='lower',extent=[0,1,0,1],cmap=cmap)
    ax.set(title=title,xlabel='x (m)',ylabel='y (m)')
    fig.colorbar(im,ax=ax,shrink=.82)
plt.show()

## 5. Controlled experiment — 20 ms versus 10 ms without changing movement

Below, both DT choices sample the **identical 5 ms reference trajectory**. To isolate binning from Poisson trial-to-trial noise, first generate **one fine-grid spike realization**, then sum these same spike events into 10 ms and 20 ms bins. The total counted spikes must be **identical** because only the bin boundaries change.

The table reports (1) the integral approximated on the fine grid, (2) the expected count from each bin-start rate approximation, and (3) the total of the **shared** simulated spikes rebinned to each DT. Small differences in the approximate expected count reflect numerical sampling of a changing rate, not new motion. If you generate fresh independent spike trains instead, their totals generally differ due to Poisson variability even at fixed trajectory.

In [ ]:
# Same continuous motion; one common Poisson realization at the fine motion step.
fine_rate = place_rate(reference_position[:-1])
fine_lambda = fine_rate*TRAJECTORY_DT
fine_spikes = np.random.default_rng(9026).poisson(fine_lambda)
fine_expected_total = float(fine_lambda.sum())
common_spike_total = int(fine_spikes.sum())

print("Spikes on the fixed path at two observation bin widths")
print("DT (s) | fine-grid expected | bin-start expected | shared observed | number of bins")
for dt_cmp in (0.02, 0.01):
    n_cmp = int(round(DURATION/dt_cmp))
    grouping = int(round(dt_cmp/TRAJECTORY_DT))
    if not np.isclose(n_cmp*dt_cmp, DURATION, atol=1e-9) or not np.isclose(grouping*TRAJECTORY_DT, dt_cmp):
        raise ValueError("Comparison DT must be an integer multiple of motion DT and divide the duration.")
    cmp_times = np.arange(n_cmp)*dt_cmp
    cmp_positions = sample_path(cmp_times, reference_times, reference_position)
    binstart_expected = float(np.sum(place_rate(cmp_positions)*dt_cmp))
    rebinned_spikes = fine_spikes.reshape(n_cmp, grouping).sum(axis=1)
    assert int(rebinned_spikes.sum()) == common_spike_total  # Same events, new bins only.
    print(f"{dt_cmp:6.3f} | {fine_expected_total:18.2f} | "
          f"{binstart_expected:18.2f} | {int(rebinned_spikes.sum()):15d} | {n_cmp:14,d}")

print("Common events are identical; trajectory and fine-step rate process are unchanged.")

## 6. Student experiments and interpretation

1. **Change field width:** set sigma to 0.25 m and compare the rate field and spatial rate map. The parameter-aware Gaussian validation should still pass.
2. **Reduce observation time:** set DURATION to 60 s and compare spatial coverage and rate-map mean absolute error. The fine motion path has the same seeded prefix.
3. **Change the spike-count bin:** set DT from 0.02 s to 0.01 s and Run All. Confirm that the underlying fine trajectory does not change, that the number of bins doubles, and that **expected total counts** are comparable. Then inspect the controlled 10/20 ms rebinning experiment in Section 5.
4. **Repeat with different seeds:** change MOTION_SEED and SPIKE_SEED separately. Which affects the path and occupancy, and which affects spike variability? Report mean and spread across several runs.
5. **Conceptual:** Why does a map driven by externally supplied position not demonstrate an autonomous robot estimating its location? Propose a population decoder and held-out evaluation.

**Limitations:** No learned place fields, remapping, refractory dynamics, LiDAR-based localization, or measured sensor uncertainty are modeled. The chosen fine integration interval approximates a continuous-time path; using an even finer reference grid is a separate numerical convergence question.